# Capital Reallocation and Risk-Adjusted Performance Analysis

## Project Goal

This notebook builds an elementary but useful framework for analyzing **capital reallocation** and **risk-adjusted performance**.

The main idea is:

> A strategy should not be judged only by how much money it makes.  
> It should also be judged by how much risk it takes to earn that return.

In this project, we compare different ways of allocating capital across several assets or strategies:

1. Equal-weight allocation  
2. Momentum-based allocation  
3. Volatility-adjusted allocation  
4. Risk-adjusted reallocation  

We then evaluate each method using:

- Total return  
- Annualized return  
- Annualized volatility  
- Sharpe ratio  
- Sortino ratio  
- Maximum drawdown  
- Calmar ratio  

This notebook is designed as an educational project and not as financial advice.

## 1. Import Libraries

We use standard Python libraries for data analysis and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

## 2. Load or Simulate Price Data

This notebook can work with your own CSV file or with simulated data.

### Option A: Use your own file

If you have a CSV file, it should contain a `date` column and several asset price columns, for example:

- `BTC`
- `ETH`
- `SPY`
- `QQQ`
- `GLD`

Each asset column should contain price values.

### Option B: Use simulated data

If no file is found, the notebook creates simulated prices for four example assets.

In [ ]:
csv_file = "multi_asset_prices.csv"

try:
    prices = pd.read_csv(csv_file)
    print(f"Loaded data from {csv_file}")

    prices.columns = [col.lower().strip().replace(" ", "_") for col in prices.columns]

    if "date" in prices.columns:
        prices["date"] = pd.to_datetime(prices["date"])
        prices = prices.sort_values("date").set_index("date")

except FileNotFoundError:
    print("No CSV file found. Creating simulated multi-asset price data.")

    np.random.seed(42)
    n = 1000
    dates = pd.date_range(start="2020-01-01", periods=n, freq="D")

    # Simulated assets with different return and risk profiles
    asset_settings = {
        "Asset_A": {"mu": 0.0006, "sigma": 0.018},
        "Asset_B": {"mu": 0.0004, "sigma": 0.012},
        "Asset_C": {"mu": 0.0008, "sigma": 0.025},
        "Asset_D": {"mu": 0.0002, "sigma": 0.008},
    }

    prices = pd.DataFrame(index=dates)

    for asset, params in asset_settings.items():
        returns = np.random.normal(params["mu"], params["sigma"], n)
        prices[asset] = 100 * (1 + pd.Series(returns, index=dates)).cumprod()

prices.head()

## 3. Inspect Price Data

We first inspect the dataset and plot the price paths.

Because assets may have different price levels, the raw price plot can be difficult to compare. Later we will normalize them.

In [ ]:
print("Shape:", prices.shape)
prices.info()

In [ ]:
prices.plot(figsize=(12, 5))
plt.title("Asset Prices")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()

## 4. Normalize Prices

Normalizing prices means starting each asset at 1. This makes performance easier to compare.

For each asset:

$$
\text{Normalized Price}_t = \frac{P_t}{P_0}
$$

In [ ]:
normalized_prices = prices / prices.iloc[0]

normalized_prices.plot(figsize=(12, 5))
plt.title("Normalized Asset Prices")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

## 5. Calculate Returns

We calculate daily percentage returns:

$$
r_t = \frac{P_t}{P_{t-1}} - 1
$$

Returns are the foundation for portfolio analysis.

In [ ]:
returns = prices.pct_change().dropna()
returns.head()

In [ ]:
returns.describe().T

## 6. Create Core Performance Metric Functions

We now define reusable functions for evaluating performance.

These functions will help us compare assets and portfolio allocation methods.

In [ ]:
def annualized_return(r, periods_per_year=252):
    r = r.dropna()
    if len(r) == 0:
        return np.nan
    growth = (1 + r).prod()
    return growth ** (periods_per_year / len(r)) - 1

def annualized_volatility(r, periods_per_year=252):
    return r.dropna().std() * np.sqrt(periods_per_year)

def sharpe_ratio(r, risk_free_rate=0.0, periods_per_year=252):
    r = r.dropna()
    ann_ret = annualized_return(r, periods_per_year)
    ann_vol = annualized_volatility(r, periods_per_year)
    if ann_vol == 0:
        return np.nan
    return (ann_ret - risk_free_rate) / ann_vol

def sortino_ratio(r, risk_free_rate=0.0, periods_per_year=252):
    r = r.dropna()
    ann_ret = annualized_return(r, periods_per_year)
    downside = r[r < 0]
    downside_vol = downside.std() * np.sqrt(periods_per_year)
    if downside_vol == 0:
        return np.nan
    return (ann_ret - risk_free_rate) / downside_vol

def max_drawdown(r):
    r = r.dropna()
    equity = (1 + r).cumprod()
    running_max = equity.cummax()
    drawdown = equity / running_max - 1
    return drawdown.min()

def calmar_ratio(r, periods_per_year=252):
    ann_ret = annualized_return(r, periods_per_year)
    mdd = abs(max_drawdown(r))
    if mdd == 0:
        return np.nan
    return ann_ret / mdd

def performance_table(return_data, periods_per_year=252):
    result = pd.DataFrame(index=return_data.columns)
    result["Total Return"] = (1 + return_data).prod() - 1
    result["Annualized Return"] = return_data.apply(annualized_return, periods_per_year=periods_per_year)
    result["Annualized Volatility"] = return_data.apply(annualized_volatility, periods_per_year=periods_per_year)
    result["Sharpe Ratio"] = return_data.apply(sharpe_ratio, periods_per_year=periods_per_year)
    result["Sortino Ratio"] = return_data.apply(sortino_ratio, periods_per_year=periods_per_year)
    result["Max Drawdown"] = return_data.apply(max_drawdown)
    result["Calmar Ratio"] = return_data.apply(calmar_ratio, periods_per_year=periods_per_year)
    return result

## 7. Analyze Individual Assets

Before building portfolios, we examine each asset separately.

This helps answer:

- Which asset had the highest return?
- Which asset had the highest risk?
- Which asset had the best risk-adjusted performance?

In [ ]:
asset_metrics = performance_table(returns)
asset_metrics.round(4)

In [ ]:
asset_metrics[["Annualized Return", "Annualized Volatility"]].plot(kind="bar", figsize=(10, 5))
plt.title("Annualized Return vs Annualized Volatility")
plt.ylabel("Value")
plt.xticks(rotation=45)
plt.show()

In [ ]:
asset_metrics[["Sharpe Ratio", "Sortino Ratio", "Calmar Ratio"]].plot(kind="bar", figsize=(10, 5))
plt.title("Risk-Adjusted Performance by Asset")
plt.ylabel("Ratio")
plt.xticks(rotation=45)
plt.show()

## 8. Equal-Weight Portfolio

The simplest capital allocation rule is equal weighting.

If there are four assets, each receives 25% of the capital.

This is a useful benchmark because it does not require forecasting.

In [ ]:
n_assets = returns.shape[1]
equal_weights = pd.Series(1 / n_assets, index=returns.columns)

equal_weights

In [ ]:
equal_weight_returns = returns.dot(equal_weights)
equal_weight_equity = (1 + equal_weight_returns).cumprod()

equal_weight_equity.plot(figsize=(12, 5))
plt.title("Equal-Weight Portfolio Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

## 9. Volatility-Adjusted Allocation

Equal weighting gives the same dollar amount to every asset.

But if one asset is much more volatile, it contributes more risk to the portfolio.

A simple risk-based idea is:

> Give less capital to high-volatility assets and more capital to low-volatility assets.

We use inverse volatility weights:

$$
w_i = \frac{1 / \sigma_i}{\sum_j 1 / \sigma_j}
$$

In [ ]:
lookback_vol = 60

rolling_vol = returns.rolling(lookback_vol).std()

inverse_vol = 1 / rolling_vol
inverse_vol_weights = inverse_vol.div(inverse_vol.sum(axis=1), axis=0)

inverse_vol_weights.tail()

In [ ]:
inverse_vol_portfolio_returns = (returns * inverse_vol_weights.shift(1)).sum(axis=1)
inverse_vol_portfolio_returns = inverse_vol_portfolio_returns.dropna()

inverse_vol_equity = (1 + inverse_vol_portfolio_returns).cumprod()

inverse_vol_equity.plot(figsize=(12, 5))
plt.title("Inverse-Volatility Portfolio Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

## 10. Momentum-Based Capital Reallocation

Momentum-based reallocation assigns more capital to assets with stronger recent performance.

Here we compute each asset's recent return over a lookback window and allocate only to assets with positive momentum.

The simple rule is:

- Calculate 60-day momentum
- Keep only assets with positive momentum
- Allocate more to assets with stronger positive momentum
- If no asset has positive momentum, hold cash

In [ ]:
lookback_momentum = 60

momentum = prices.pct_change(lookback_momentum)

positive_momentum = momentum.clip(lower=0)

momentum_weights = positive_momentum.div(positive_momentum.sum(axis=1), axis=0)
momentum_weights = momentum_weights.fillna(0)

momentum_weights.tail()

In [ ]:
momentum_portfolio_returns = (returns * momentum_weights.shift(1)).sum(axis=1)
momentum_portfolio_returns = momentum_portfolio_returns.dropna()

momentum_equity = (1 + momentum_portfolio_returns).cumprod()

momentum_equity.plot(figsize=(12, 5))
plt.title("Momentum-Based Reallocation Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

## 11. Risk-Adjusted Momentum Allocation

Raw momentum can over-allocate to assets that have strong returns but very high risk.

A more careful version uses risk-adjusted momentum:

$$
\text{Risk-Adjusted Momentum} = \frac{\text{Recent Return}}{\text{Recent Volatility}}
$$

This is similar in spirit to rewarding assets that produce more return per unit of risk.

In [ ]:
recent_return = prices.pct_change(lookback_momentum)
recent_vol = returns.rolling(lookback_momentum).std()

risk_adjusted_score = recent_return / recent_vol
risk_adjusted_score = risk_adjusted_score.clip(lower=0)

risk_adjusted_weights = risk_adjusted_score.div(risk_adjusted_score.sum(axis=1), axis=0)
risk_adjusted_weights = risk_adjusted_weights.fillna(0)

risk_adjusted_weights.tail()

In [ ]:
risk_adjusted_portfolio_returns = (returns * risk_adjusted_weights.shift(1)).sum(axis=1)
risk_adjusted_portfolio_returns = risk_adjusted_portfolio_returns.dropna()

risk_adjusted_equity = (1 + risk_adjusted_portfolio_returns).cumprod()

risk_adjusted_equity.plot(figsize=(12, 5))
plt.title("Risk-Adjusted Momentum Portfolio Equity Curve")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

## 12. Combine Portfolio Returns

Now we combine all portfolio return series into one dataframe for comparison.

In [ ]:
portfolio_returns = pd.DataFrame({
    "Equal Weight": equal_weight_returns,
    "Inverse Volatility": inverse_vol_portfolio_returns,
    "Momentum": momentum_portfolio_returns,
    "Risk-Adjusted Momentum": risk_adjusted_portfolio_returns
}).dropna()

portfolio_returns.head()

In [ ]:
portfolio_equity = (1 + portfolio_returns).cumprod()

portfolio_equity.plot(figsize=(12, 5))
plt.title("Portfolio Strategy Comparison")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

## 13. Portfolio Performance Table

This table compares the strategies using return and risk-adjusted metrics.

Important interpretation:

- Higher annualized return is better, but only if risk is acceptable.
- Lower volatility means smoother returns.
- Higher Sharpe ratio means better return per unit of total volatility.
- Higher Sortino ratio means better return per unit of downside volatility.
- Lower maximum drawdown means smaller peak-to-trough loss.
- Higher Calmar ratio means better return relative to drawdown risk.

In [ ]:
portfolio_metrics = performance_table(portfolio_returns)
portfolio_metrics.round(4)

## 14. Visualize Risk-Adjusted Metrics

We now compare Sharpe, Sortino, and Calmar ratios across the allocation methods.

In [ ]:
portfolio_metrics[["Sharpe Ratio", "Sortino Ratio", "Calmar Ratio"]].plot(kind="bar", figsize=(10, 5))
plt.title("Risk-Adjusted Performance Comparison")
plt.ylabel("Ratio")
plt.xticks(rotation=30)
plt.show()

## 15. Drawdown Comparison

Drawdown is often more emotionally and practically important than volatility.

A strategy may have strong average returns but still be difficult to trade if it experiences large drawdowns.

In [ ]:
drawdowns = pd.DataFrame(index=portfolio_returns.index)

for col in portfolio_returns.columns:
    equity = (1 + portfolio_returns[col]).cumprod()
    running_max = equity.cummax()
    drawdowns[col] = equity / running_max - 1

drawdowns.plot(figsize=(12, 5))
plt.title("Portfolio Drawdown Comparison")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.show()

## 16. Rolling Sharpe Ratio

Performance changes over time.

A rolling Sharpe ratio shows whether a strategy's risk-adjusted performance is stable or unstable.

In [ ]:
rolling_window = 126

rolling_sharpe = portfolio_returns.rolling(rolling_window).mean() / portfolio_returns.rolling(rolling_window).std()
rolling_sharpe = rolling_sharpe * np.sqrt(252)

rolling_sharpe.plot(figsize=(12, 5))
plt.title("Rolling Sharpe Ratio")
plt.xlabel("Date")
plt.ylabel("Rolling Sharpe")
plt.show()

## 17. Capital Allocation Over Time

For dynamic strategies, the weights change through time.

We visualize the weights for:

- Inverse-volatility allocation
- Momentum allocation
- Risk-adjusted momentum allocation

In [ ]:
inverse_vol_weights.dropna().plot(figsize=(12, 5))
plt.title("Inverse-Volatility Weights Over Time")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.show()

In [ ]:
momentum_weights.loc[portfolio_returns.index].plot(figsize=(12, 5))
plt.title("Momentum Weights Over Time")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.show()

In [ ]:
risk_adjusted_weights.loc[portfolio_returns.index].plot(figsize=(12, 5))
plt.title("Risk-Adjusted Momentum Weights Over Time")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.show()

## 18. Add Basic Rebalancing Costs

Capital reallocation is not free.

Whenever portfolio weights change, the strategy may generate transaction costs.

We estimate turnover as:

$$
\text{Turnover}_t = \sum_i |w_{i,t} - w_{i,t-1}|
$$

Then we subtract a cost proportional to turnover.

In [ ]:
transaction_cost_rate = 0.001  # 0.10% per full turnover

def apply_rebalancing_costs(asset_returns, weights, transaction_cost_rate=0.001):
    aligned_weights = weights.reindex(asset_returns.index).fillna(0)
    shifted_weights = aligned_weights.shift(1).fillna(0)

    gross_returns = (asset_returns * shifted_weights).sum(axis=1)
    turnover = aligned_weights.diff().abs().sum(axis=1).fillna(0)
    costs = turnover * transaction_cost_rate

    net_returns = gross_returns - costs

    return net_returns, turnover, costs

momentum_net_returns, momentum_turnover, momentum_costs = apply_rebalancing_costs(
    returns, momentum_weights, transaction_cost_rate
)

ra_net_returns, ra_turnover, ra_costs = apply_rebalancing_costs(
    returns, risk_adjusted_weights, transaction_cost_rate
)

cost_adjusted_returns = pd.DataFrame({
    "Momentum Gross": momentum_portfolio_returns,
    "Momentum Net": momentum_net_returns,
    "Risk-Adjusted Momentum Gross": risk_adjusted_portfolio_returns,
    "Risk-Adjusted Momentum Net": ra_net_returns
}).dropna()

cost_adjusted_equity = (1 + cost_adjusted_returns).cumprod()

cost_adjusted_equity.plot(figsize=(12, 5))
plt.title("Effect of Rebalancing Costs")
plt.xlabel("Date")
plt.ylabel("Growth of $1")
plt.show()

In [ ]:
cost_adjusted_metrics = performance_table(cost_adjusted_returns)
cost_adjusted_metrics.round(4)

## 19. Main Findings Template

After running the notebook, summarize your findings using this structure:

1. **Best total return:**  
   Which allocation method produced the highest total return?

2. **Best Sharpe ratio:**  
   Which method produced the best return per unit of volatility?

3. **Best Sortino ratio:**  
   Which method handled downside risk best?

4. **Lowest drawdown:**  
   Which method had the smallest peak-to-trough loss?

5. **Most stable strategy:**  
   Which method had the most consistent rolling Sharpe ratio?

6. **Effect of costs:**  
   Did transaction costs meaningfully reduce the performance of dynamic reallocation methods?

## 20. Interpretation

Capital reallocation is a central idea in portfolio management.

Instead of asking only which asset is best, we ask:

> How much capital should be assigned to each asset at each point in time?

This notebook compared several allocation philosophies:

- **Equal weight:** simple and stable
- **Inverse volatility:** gives less capital to riskier assets
- **Momentum:** follows recent winners
- **Risk-adjusted momentum:** follows recent winners after accounting for volatility

The most important lesson is that return alone is not enough. A portfolio with lower return but much lower drawdown or higher Sharpe ratio may be more attractive than a portfolio with higher raw return but extreme risk.

## 21. Limitations

This project is intentionally simplified.

Important limitations include:

- Simulated data may not reflect real market behavior.
- Transaction costs are simplified.
- The model does not include taxes, slippage, liquidity, or market impact.
- The strategy uses fixed lookback windows.
- There is no walk-forward optimization.
- There is no out-of-sample validation.
- The analysis assumes daily rebalancing unless modified.
- Risk-adjusted performance can change across market regimes.

A professional version would require more rigorous testing.

## 22. Possible Extensions

This notebook can be extended in several ways:

1. Use real data from stocks, ETFs, or cryptocurrencies.
2. Add monthly or weekly rebalancing instead of daily rebalancing.
3. Add volatility targeting at the portfolio level.
4. Add maximum position size constraints.
5. Add a cash or Treasury bill asset.
6. Add rolling correlation analysis.
7. Add optimization methods such as minimum variance or maximum Sharpe portfolios.
8. Add walk-forward testing.
9. Add Monte Carlo simulation.
10. Compare results across different market regimes.

## 23. Final Conclusion

This notebook developed a practical framework for capital reallocation and risk-adjusted performance analysis.

The main takeaway is:

> A strong portfolio strategy is not simply the one with the highest return.  
> It is the one that produces attractive returns while controlling volatility, downside risk, and drawdowns.

By comparing equal weight, inverse volatility, momentum, and risk-adjusted momentum allocation, this project shows how capital allocation rules can strongly affect portfolio outcomes.

This framework can serve as a foundation for more advanced portfolio construction, systematic trading, and quantitative investment research.